In [ ]:
import mlflow
import nannyml as nml
import pandas as pd
from dotenv import load_dotenv
from nannyml.thresholds import ConstantThreshold, StandardDeviationThreshold

load_dotenv("../config/.env")
mlflow.set_registry_uri("databricks-uc")

In [ ]:
model_uri = "models:/betsim.models.xgboost_optuna@champion"
model = mlflow.xgboost.load_model(model_uri)

In [ ]:
df = pd.read_parquet("../data/processed/jleague.parquet")
df_reference = df.query("fold == 'development'")
df_analysis = df.query("fold == 'test'")

resp_var = "hcap_res"
exp_vars = model.get_booster().feature_names

for df in [df_reference, df_analysis]:
    df["pi_hat"] = model.predict_proba(df[exp_vars])[:, 1]
    df["y_hat"] = model.predict(df[exp_vars])

#### Confidence-Based Performance Estimation

In [ ]:
thld_const = ConstantThreshold(lower=0)
thld_std = StandardDeviationThreshold(
    std_lower_multiplier=2,
    std_upper_multiplier=2,
)

In [ ]:
estimator = nml.CBPE(
    metrics=["roc_auc", "f1", "accuracy"],
    y_pred_proba="pi_hat",
    y_true=resp_var,
    problem_type="classification_binary",
    y_pred="y_hat",
    timestamp_column_name="game_dt",
    chunk_period="m",
    thresholds={
        "roc_auc": thld_std,
        "f1": thld_std,
        "accuracy": thld_std,
    },
)

estimator.fit(df_reference)
est_perf = estimator.estimate(df_analysis)

In [ ]:
est_perf.plot().show()

#### Performance Realisation

In [ ]:
calculator = nml.PerformanceCalculator(
    metrics=["roc_auc", "f1", "accuracy", "business_value"],
    y_true=resp_var,
    problem_type="classification_binary",
    y_pred="y_hat",
    y_pred_proba="pi_hat",
    timestamp_column_name="game_dt",
    thresholds={
        "roc_auc": thld_std,
        "f1": thld_std,
        "accuracy": thld_std,
        "business_value": thld_const,
    },
    chunk_period="m",
    business_value_matrix=[[360, -200], [-200, 360]],
)

calculator.fit(df_reference)
rls_perf = calculator.calculate(df_analysis)

In [ ]:
for metric in ["roc_auc", "f1", "accuracy"]:
    rls_perf.filter(metrics=metric) \
        .compare(est_perf.filter(metrics=metric)) \
        .plot().show()

rls_perf.filter(period="analysis", metrics="business_value").plot().show()

#### Multivariate Drift Detection

In [ ]:
calculator = nml.DataReconstructionDriftCalculator(
    column_names=exp_vars,
    timestamp_column_name="game_dt",
    chunk_period="m",
	threshold=thld_std,
)

calculator.fit(df_reference)
results = calculator.calculate(df_analysis)

In [ ]:
results.filter(period="analysis").plot().show()